# Generating the results from the reduced model with derived variables, minus the delphi and household alert count

In [1]:
# Opening the datasets

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)

# Locate the project folder.
relative_folder = Path("data") / "KGB Modelling"

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_folder / "KGB in sample.csv").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the KGB modelling data.")

data_folder = project_root / relative_folder

# Load the development and validation samples.
kgb_in_sample = pd.read_csv(data_folder / "KGB in sample.csv")
kgb_out_sample = pd.read_csv(data_folder / "KGB out sample.csv")

target = "target_bad"

# Check the target is complete and binary.
for name, data in [
    ("In-sample", kgb_in_sample),
    ("Out-of-sample", kgb_out_sample),
]:
    if target not in data.columns:
        raise ValueError(f"{name}: missing {target}.")

    if not data[target].isin([0, 1]).all():
        raise ValueError(f"{name}: {target} must contain only 0 and 1.")

    if data[target].nunique() != 2:
        raise ValueError(f"{name}: both good and bad accounts are required.")

# Review sample sizes and bad rates.
sample_summary = pd.DataFrame([
    {
        "Sample": name,
        "Accounts": len(data),
        "Bads": int(data[target].sum()),
        "Bad rate": data[target].mean(),
    }
    for name, data in [
        ("In-sample", kgb_in_sample),
        ("Out-of-sample", kgb_out_sample),
    ]
])

In [2]:
# Creating the variables

def add_derived_variables(data):
    data = data.copy()

    required_columns = [
        "disposable_income",
        "time_at_address_months",
        "employment_tenure_months",
    ]

    missing_columns = set(required_columns) - set(data.columns)
    if missing_columns:
        raise ValueError(f"Missing columns: {sorted(missing_columns)}")

    for column in required_columns:
        data[column] = pd.to_numeric(data[column], errors="raise")

        if data[column].isin([np.inf, -np.inf]).any():
            raise ValueError(f"{column} contains infinite values.")

    for column in [
        "time_at_address_months",
        "employment_tenure_months",
    ]:
        if data[column].dropna().lt(0).any():
            raise ValueError(f"{column} contains negative tenure.")

    # Nullable numbers preserve missing inputs in comparisons.
    surplus = data["disposable_income"].astype("Float64")
    address_tenure = data["time_at_address_months"].astype("Float64")
    employment_tenure = data["employment_tenure_months"].astype("Float64")

    # 1 = disposable income below £150; 0 = £150 or above.
    data["low_disposable_income_flag"] = surplus.lt(150).astype("Int64")

    # Natural logarithm; zero months becomes zero.
    data["log_address_tenure"] = np.log1p(address_tenure)

    # Count of employment/address tenures below 12 months.
    data["employment_address_instability_count"] = (
        employment_tenure.lt(12).astype("Int64")
        + address_tenure.lt(12).astype("Int64")
    )

    return data


# Apply identical definitions to both samples.
kgb_in_sample = add_derived_variables(kgb_in_sample)
kgb_out_sample = add_derived_variables(kgb_out_sample)

derived_columns = [
    "low_disposable_income_flag",
    "log_address_tenure",
    "employment_address_instability_count",
]


---

# Running the reduced model

In [5]:
from sklearn.metrics import roc_curve

# Find the data whether VS Code is running from the project or notebooks folder.
relative_dir = Path("data") / "KGB Modelling"
project_root = next(
    (
        folder for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_dir / "KGB in sample.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/KGB Modelling.")

data_dir = project_root / relative_dir

# These are the predictors retained in the reduced model.
scaled_predictors = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "log_address_tenure",
]
numeric_inputs = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "disposable_income",
    "time_at_address_months",
]
required_columns = ["target_bad", "loan_purpose", *numeric_inputs]


def load_sample(filename):
    data = pd.read_csv(data_dir / filename, usecols=required_columns)
    rows = data.replace(r"^\s*$", np.nan, regex=True).copy()

    for column in ["target_bad", *numeric_inputs]:
        rows[column] = pd.to_numeric(rows[column], errors="raise")

    rows = rows.replace([np.inf, -np.inf], np.nan).dropna().copy()

    if not rows["target_bad"].isin([0, 1]).all():
        raise ValueError(f"{filename}: target_bad must be 0 or 1.")
    if rows["target_bad"].nunique() != 2:
        raise ValueError(f"{filename}: both goods and bads are required.")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError(f"{filename}: address tenure cannot be negative.")

    rows["low_disposable_income_flag"] = (
        rows["disposable_income"] < 150
    ).astype(float)
    rows["log_address_tenure"] = np.log1p(
        rows["time_at_address_months"]
    )

    print(
        f"{filename}: retained {len(rows):,}; "
        f"excluded {len(data) - len(rows):,}"
    )
    return rows


train = load_sample("KGB in sample.csv")
test = load_sample("KGB out sample.csv")

# Learn the scaling parameters from the development sample only.
train_means = train[scaled_predictors].mean()
train_stds = train[scaled_predictors].std(ddof=0)
if not np.isfinite(train_stds).all() or train_stds.eq(0).any():
    raise ValueError("A training predictor has invalid or zero variation.")


def design_matrix(rows):
    X = (rows[scaled_predictors] - train_means) / train_stds
    X.insert(0, "const", 1.0)
    X["low_disposable_income_flag"] = rows["low_disposable_income_flag"]
    X["loan_purpose=education"] = (
        rows["loan_purpose"] == "education"
    ).astype(float)
    return X.astype(float)


X_train = design_matrix(train)
X_test = design_matrix(test)
y_train = train["target_bad"].astype(int)
y_test = test["target_bad"].astype(int)

# Fit on development data; score both samples.
reduced_model = sm.Logit(y_train, X_train).fit(
    maxiter=200, disp=False
)
if not reduced_model.mle_retvals.get("converged", False):
    raise RuntimeError("Reduced model did not converge.")

train_pd = pd.Series(reduced_model.predict(X_train), index=train.index)
test_pd = pd.Series(reduced_model.predict(X_test), index=test.index)


def performance(actual, probability):
    auc = roc_auc_score(actual, probability)
    fpr, tpr, _ = roc_curve(actual, probability)
    return {
        "Accounts": len(actual),
        "Observed bad rate": actual.mean(),
        "Mean predicted PD": probability.mean(),
        "ROC AUC": auc,
        "Gini": 2 * auc - 1,
        "KS": np.max(tpr - fpr),
    }


results = pd.DataFrame({
    "In-sample": performance(y_train, train_pd),
    "Out-of-sample": performance(y_test, test_pd),
})
display(results.round(4))
print(reduced_model.summary())

# Select the KS-maximising cutoff on development data only.
fpr, tpr, thresholds = roc_curve(
    y_train, train_pd, drop_intermediate=False
)
finite = np.isfinite(thresholds)
best_index = np.flatnonzero(finite)[
    np.argmax((tpr - fpr)[finite])
]

KGB in sample.csv: retained 7,258; excluded 0
KGB out sample.csv: retained 2,458; excluded 0


,In-sample,Out-of-sample
Accounts,7258.0000,2458.0000
Observed bad rate,0.0405,0.0415
Mean predicted PD,0.0405,0.0391
ROC AUC,0.6407,0.6442
Gini,0.2814,0.2884
KS,0.2288,0.2201


                           Logit Regression Results                           
Dep. Variable:             target_bad   No. Observations:                 7258
Model:                          Logit   Df Residuals:                     7250
Method:                           MLE   Df Model:                            7
Date:                Tue, 29 Sep 2026   Pseudo R-squ.:                 0.02744
Time:                        15:58:45   Log-Likelihood:                -1196.8
converged:                       True   LL-Null:                       -1230.6
Covariance Type:            nonrobust   LLR p-value:                 4.634e-12
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -3.3139      0.075    -44.195      0.000      -3.461      -3.167
debt_to_income_ratio           0.1063      0.058      1.838      0.066      -0.007 